In [2]:
import copy
from heapq import heappush, heappop

n = 3
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]


class PriorityQueue:
    def __init__(self):
        self.heap = []

    def push(self, key):
        heappush(self.heap, key)

    def pop(self):
        return heappop(self.heap)

    def empty(self):
        return not self.heap


class Node:
    def __init__(self, parent, mats, empty_pos, cost, level):
        self.parent = parent
        self.mats = mats
        self.empty_pos = empty_pos
        self.cost = cost  # h (heuristic)
        self.level = level  # g (chi phí từ gốc)
        self.f = cost + level  # f = g + h

    def __lt__(self, other):
        return self.f < other.f


def manhattan_distance(mats, final):
    dist = 0
    for i in range(n):
        for j in range(n):
            if mats[i][j] != 0:
                # Tìm vị trí đích của số này
                for x in range(n):
                    for y in range(n):
                        if final[x][y] == mats[i][j]:
                            dist += abs(i - x) + abs(j - y)
                            break
    return dist


def new_node(mats, empty_pos, new_empty_pos, level, parent, final):
    new_mats = copy.deepcopy(mats)
    x1, y1 = empty_pos
    x2, y2 = new_empty_pos
    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    cost = manhattan_distance(new_mats, final)
    return Node(parent, new_mats, new_empty_pos, cost, level)


def print_matrix(mats):
    for row in mats:
        print(" ".join(f"{x:2d}" for x in row))
    print()


def print_path(root):
    if root is None:
        return
    print_path(root.parent)
    print_matrix(root.mats)


def solve_8puzzle(initial, empty_pos, final):
    pq = PriorityQueue()
    root = Node(None, initial, empty_pos, manhattan_distance(initial, final), 0)
    pq.push(root)

    visited = set()  # Để tránh lặp trạng thái (tùy chọn nhưng rất nên thêm)

    while not pq.empty():
        current = pq.pop()

        # Chuyển ma trận thành tuple để hash
        state = tuple(tuple(row) for row in current.mats)
        if state in visited:
            continue
        visited.add(state)

        if current.cost == 0:  # Đã đến đích
            print("Đường đi tìm được (số bước =", current.level, "):")
            print_path(current)
            return

        for i in range(4):
            new_pos = [current.empty_pos[0] + rows[i], current.empty_pos[1] + cols[i]]
            if 0 <= new_pos[0] < n and 0 <= new_pos[1] < n:
                child = new_node(
                    current.mats,
                    current.empty_pos,
                    new_pos,
                    current.level + 1,
                    current,
                    final,
                )
                pq.push(child)

    print("Không tìm thấy lời giải!")


# ================== Chạy ==================
initial = [[1, 2, 3], [5, 6, 0], [7, 8, 4]]

final = [[1, 2, 3], [5, 8, 6], [0, 7, 4]]

empty_tile_pos = [1, 2]

solve_8puzzle(initial, empty_tile_pos, final)


Đường đi tìm được (số bước = 3 ):
 1  2  3
 5  6  0
 7  8  4

 1  2  3
 5  0  6
 7  8  4

 1  2  3
 5  8  6
 7  0  4

 1  2  3
 5  8  6
 0  7  4

